In [1]:
from pyspark.sql import functions as F


In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
   .appName("Colab Spark Example") \
   .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
   .getOrCreate()


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/27 11:45:08 WARN Utils: Your hostname, Marcins-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.0.2.114 instead (on interface en0)
26/06/27 11:45:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/27 11:45:08 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/27 11:45:09 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [10]:
path = "../data/"

inDF = spark.read.format("csv") \
 .option("sep", ",") \
 .option("inferSchema", "true") \
 .option("spark.executor.memory", "4g") \
 .option("spark.driver.memory", "4g") \
 .option("header", "true") \
 .load(path + "/US_Accidents_March23.csv")

inDF.printSchema()


root
 |-- ID: string (nullable = true)
 |-- Source: string (nullable = true)
 |-- Severity: integer (nullable = true)
 |-- Start_Time: timestamp (nullable = true)
 |-- End_Time: timestamp (nullable = true)
 |-- Start_Lat: double (nullable = true)
 |-- Start_Lng: double (nullable = true)
 |-- End_Lat: double (nullable = true)
 |-- End_Lng: double (nullable = true)
 |-- Distance(mi): double (nullable = true)
 |-- Description: string (nullable = true)
 |-- Street: string (nullable = true)
 |-- City: string (nullable = true)
 |-- County: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Zipcode: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- Timezone: string (nullable = true)
 |-- Airport_Code: string (nullable = true)
 |-- Weather_Timestamp: timestamp (nullable = true)
 |-- Temperature(F): double (nullable = true)
 |-- Wind_Chill(F): double (nullable = true)
 |-- Humidity(%): double (nullable = true)
 |-- Pressure(in): double (nullable = true)
 |-- V

In [4]:
inDF.count()

7728394

In [6]:
inDF.groupBy("State").count().show()


+-----+-------+
|State|  count|
+-----+-------+
|   SC| 382557|
|   NJ| 140719|
|   DC|  18630|
|   OR| 179660|
|   VA| 303301|
|   RI|  16971|
|   KY|  32254|
|   NH|  10213|
|   MI| 162191|
|   WI|  34688|
|   CA|1741433|
|   NE|  28870|
|   CT|  71005|
|   MD| 140417|
|   DE|  14097|
|   MO|  77323|
|   IL| 168958|
|   WA| 108221|
|   IN|  67224|
|   OH| 118115|
+-----+-------+
only showing top 20 rows


In [8]:
inDF.select(F.countDistinct("City")).show()


+--------------------+
|count(DISTINCT City)|
+--------------------+
|               13678|
+--------------------+



In [11]:
from pyspark.sql.window import Window
from pyspark.sql.functions import count, row_number

# Grupujemy dane: ile wypadków dla danej pogody w danym stanie
grouped = inDF.groupBy("State", "Weather_Condition") \
 .agg(count("*") \
 .alias("count"))


In [13]:
grouped.show()

+-----+--------------------+-----+
|State|   Weather_Condition|count|
+-----+--------------------+-----+
|   IL|             Drizzle|  109|
|   TX|               Smoke|   11|
|   FL|             Drizzle|  180|
|   PA|            Overcast|18205|
|   PA|Heavy Thunderstor...|   33|
|   IL|                Mist|   75|
|   RI|       Light Drizzle|   74|
|   MD|  Light Rain / Windy|   58|
|   TX|             Thunder| 1137|
|   SC|       Partly Cloudy|31807|
|   DC|                 Fog|  126|
|   SC|                NULL| 7155|
|   CT|            Overcast| 3659|
|   NH|    Scattered Clouds|  387|
|   NE|  Light Freezing Fog|    7|
|   NH|                 Fog|  100|
|   WA|            Overcast|12850|
|   NY|                Mist|  195|
|   MA|                Rain|  838|
|   OH|      Patches of Fog|  134|
+-----+--------------------+-----+
only showing top 20 rows


In [14]:
windowSpec = Window.partitionBy("State").orderBy(grouped["count"].desc())

In [15]:
ranked = grouped.withColumn("rank", row_number().over(windowSpec))


In [16]:
top_weather_per_state = ranked.filter(ranked["rank"] == 1)
top_weather_per_state.show(50)


+-----+-----------------+------+----+
|State|Weather_Condition| count|rank|
+-----+-----------------+------+----+
|   AL|             Fair| 36994|   1|
|   AR|             Fair|  9816|   1|
|   AZ|             Fair| 86883|   1|
|   CA|             Fair|772394|   1|
|   CO|             Fair| 25343|   1|
|   CT|             Fair| 25615|   1|
|   DC|             Fair|  5903|   1|
|   DE|             Fair|  6106|   1|
|   FL|             Fair|316249|   1|
|   GA|             Fair| 45459|   1|
|   IA|             Fair|  7301|   1|
|   ID|             Fair|  5051|   1|
|   IL|             Fair| 36407|   1|
|   IN|             Fair| 17179|   1|
|   KS|             Fair|  8196|   1|
|   KY|             Fair|  7215|   1|
|   LA|             Fair| 53756|   1|
|   MA|             Fair| 14923|   1|
|   MD|             Fair| 38856|   1|
|   ME|             Fair|   462|   1|
|   MI|             Fair| 29110|   1|
|   MN|             Fair| 58458|   1|
|   MO|             Fair| 25733|   1|
|   MS|     

In [17]:
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

def upper_case(text):
   return text.upper() if text else None


In [18]:
upper_case("hello world")  # Example usage of the function

'HELLO WORLD'

In [19]:
# Rejestrujemy funkcję jako UDF
upper_udf = udf(upper_case, StringType())

# Używamy w DataFrame
inDF.withColumn("Weather_Condition_Upper", upper_udf(inDF["Weather_Condition"])).show()


26/06/27 11:57:10 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+----+-------+--------+-------------------+-------------------+------------------+------------------+-------+-------+------------+--------------------+--------------------+------------+----------+-----+----------+-------+----------+------------+-------------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+-----------------+-------+-----+--------+--------+--------+-------+-------+----------+-------+-----+---------------+--------------+------------+--------------+--------------+-----------------+---------------------+-----------------------+
|  ID| Source|Severity|         Start_Time|           End_Time|         Start_Lat|         Start_Lng|End_Lat|End_Lng|Distance(mi)|         Description|              Street|        City|    County|State|   Zipcode|Country|  Timezone|Airport_Code|  Weather_Timestamp|Temperature(F)|Wind_Chill(F)|Humidity(%)|Pressure(in)|Visibility(mi)|Wind_Direction|Wind_Speed(mph)|Precipitation

In [21]:
df2 = inDF.withColumn("stała", F.lit(1))
df2.show()

+----+-------+--------+-------------------+-------------------+------------------+------------------+-------+-------+------------+--------------------+--------------------+------------+----------+-----+----------+-------+----------+------------+-------------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+-----------------+-------+-----+--------+--------+--------+-------+-------+----------+-------+-----+---------------+--------------+------------+--------------+--------------+-----------------+---------------------+-----+
|  ID| Source|Severity|         Start_Time|           End_Time|         Start_Lat|         Start_Lng|End_Lat|End_Lng|Distance(mi)|         Description|              Street|        City|    County|State|   Zipcode|Country|  Timezone|Airport_Code|  Weather_Timestamp|Temperature(F)|Wind_Chill(F)|Humidity(%)|Pressure(in)|Visibility(mi)|Wind_Direction|Wind_Speed(mph)|Precipitation(in)|Weather_Condi

In [25]:
df2 = inDF.withColumn("Start_Lat", F.col("Start_Lat").cast("decimal(10,2)"))
df2.show()

+----+-------+--------+-------------------+-------------------+---------+------------------+-------+-------+------------+--------------------+--------------------+------------+----------+-----+----------+-------+----------+------------+-------------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+-----------------+-------+-----+--------+--------+--------+-------+-------+----------+-------+-----+---------------+--------------+------------+--------------+--------------+-----------------+---------------------+
|  ID| Source|Severity|         Start_Time|           End_Time|Start_Lat|         Start_Lng|End_Lat|End_Lng|Distance(mi)|         Description|              Street|        City|    County|State|   Zipcode|Country|  Timezone|Airport_Code|  Weather_Timestamp|Temperature(F)|Wind_Chill(F)|Humidity(%)|Pressure(in)|Visibility(mi)|Wind_Direction|Wind_Speed(mph)|Precipitation(in)|Weather_Condition|Amenity| Bump|Cross

In [26]:
df2 = inDF.withColumn("czas_minuty",
                   (F.unix_timestamp("End_Time") - F.unix_timestamp("Start_Time")) / 60)

In [28]:
df2.show()

+----+-------+--------+-------------------+-------------------+------------------+------------------+-------+-------+------------+--------------------+--------------------+------------+----------+-----+----------+-------+----------+------------+-------------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+-----------------+-------+-----+--------+--------+--------+-------+-------+----------+-------+-----+---------------+--------------+------------+--------------+--------------+-----------------+---------------------+-----------+
|  ID| Source|Severity|         Start_Time|           End_Time|         Start_Lat|         Start_Lng|End_Lat|End_Lng|Distance(mi)|         Description|              Street|        City|    County|State|   Zipcode|Country|  Timezone|Airport_Code|  Weather_Timestamp|Temperature(F)|Wind_Chill(F)|Humidity(%)|Pressure(in)|Visibility(mi)|Wind_Direction|Wind_Speed(mph)|Precipitation(in)|Weather

In [29]:
df2 = inDF.withColumn("pora_dnia",
                   F.when(F.col("Sunrise_Sunset") == "Day", "Dzień")
                    .otherwise("Noc"))
df2.show()

+----+-------+--------+-------------------+-------------------+------------------+------------------+-------+-------+------------+--------------------+--------------------+------------+----------+-----+----------+-------+----------+------------+-------------------+--------------+-------------+-----------+------------+--------------+--------------+---------------+-----------------+-----------------+-------+-----+--------+--------+--------+-------+-------+----------+-------+-----+---------------+--------------+------------+--------------+--------------+-----------------+---------------------+---------+
|  ID| Source|Severity|         Start_Time|           End_Time|         Start_Lat|         Start_Lng|End_Lat|End_Lng|Distance(mi)|         Description|              Street|        City|    County|State|   Zipcode|Country|  Timezone|Airport_Code|  Weather_Timestamp|Temperature(F)|Wind_Chill(F)|Humidity(%)|Pressure(in)|Visibility(mi)|Wind_Direction|Wind_Speed(mph)|Precipitation(in)|Weather_C

In [30]:
inDF.groupBy("Severity") \
  .avg() \
  .orderBy(F.desc("avg(Temperature(F))")).show()


+--------+-------------+------------------+------------------+------------------+-------------------+-------------------+-------------------+------------------+-----------------+------------------+-------------------+--------------------+----------------------+
|Severity|avg(Severity)|    avg(Start_Lat)|    avg(Start_Lng)|      avg(End_Lat)|       avg(End_Lng)|  avg(Distance(mi))|avg(Temperature(F))|avg(Wind_Chill(F))| avg(Humidity(%))| avg(Pressure(in))|avg(Visibility(mi))|avg(Wind_Speed(mph))|avg(Precipitation(in))|
+--------+-------------+------------------+------------------+------------------+-------------------+-------------------+-------------------+------------------+-----------------+------------------+-------------------+--------------------+----------------------+
|       1|          1.0|   35.656207071312|-94.54536974730537| 35.65551408388285|-101.07033671665549|0.11450731824183097|  72.41767236174837| 72.24167625888238|61.90745194125565|29.206987832356965|  9.4753786635757

In [32]:
inDF.groupBy("Severity") \
   .count() \
   .orderBy(F.desc("count")).show()


+--------+-------+
|Severity|  count|
+--------+-------+
|       2|6156981|
|       3|1299337|
|       4| 204710|
|       1|  67366|
+--------+-------+



In [ ]:
state_stats = inDF.groupBy("State") \
   .count() \
   .filter(F.col("count") > 10000) \
   .orderBy(F.desc("count"))
